In [4]:
"""
This file contain functions used in NPSC method. 
Mar 3rd 2024: 
All piecewiseGQ2D functions are replaced by PiecewiseGQ2D_weights_points function. 
Use global integration points and weights 
Use a global coefficient function, target function, rhs function 
"""
import torch
import numpy as np
import sympy as sp
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import math 
import time
import sys
from scipy.sparse import linalg
from pathlib import Path
if torch.cuda.is_available():  
    device = "cuda" 
else:  
    device = "cpu"  
from scipy.stats import qmc
from multiprocessing.pool import Pool
from multiprocessing.pool import ThreadPool

pi = torch.tensor(np.pi,dtype=torch.float64)
torch.set_default_dtype(torch.float64)


## models 

In [5]:

class model(nn.Module):
    """ ReLU k shallow neural network
    Parameters: 
    input size: input dimension
    hidden_size1 : number of hidden layers 
    num_classes: output classes 
    k: degree of relu functions
    """
    def __init__(self, input_size, hidden_size1, num_classes,k = 1):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, num_classes,bias = False)
        self.k = k 
    def forward(self, x):
        u1 = self.fc2(F.relu(self.fc1(x))**self.k)
        return u1
    
#neural networks with other activation functions 


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        if activation == 'tanh':
            nn.init.uniform_(m.weight, a = -R_m, b = R_m)
            nn.init.uniform_(m.bias, a = -R_m, b = R_m)
        if activation == 'relu':
            nn.init.uniform_(m.weight, a = 1, b = 1)
            nn.init.uniform_(m.bias, a = -1 , b = 1.2)
            nodes = -m.bias.data.squeeze()/m.weight.data.squeeze()
            if len(nodes[nodes < -1]) == 0: 
                print("all nodes are within the interval")
                m.bias.data[0] = 1.2 

def weights_init_uniform_non_relu(my_model,R):
    nn.init.uniform_(my_model.fc1.weight, a = -R, b = R) 
    nn.init.uniform_(my_model.fc1.bias, a = -R, b = R) 
    return my_model 

# network definition
class Network(nn.Module):
    def __init__(self, d, M):
        super(Network, self).__init__()
        self.fc_layer = nn.Sequential(nn.Linear(d, M, bias=True),nn.Tanh())
        self.output_layer = nn.Linear(M, 1, bias = False)
        
    def forward(self, x):
        h = self.fc_layer(x)
        out = self.output_layer(h)
        return out

# def Gaussian_activation(x):
#     return torch.sum(torch.exp(-x**2), dim =1,keepdim = True)
def Gaussian_activation(x):
    return torch.exp(-x**2)

def Gaussian_activation_dx(x):
    return -2*x*torch.exp(-x**2) 

class model_gaussian(nn.Module):
    """ gaussian shallow neural network
    Parameters: 
    input size: input dimension
    hidden_size1 : number of hidden layers 
    num_classes: output classes 
    """
    def __init__(self, input_size, hidden_size1, num_classes,k = 1):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, num_classes,bias = False)
    def forward(self, x):
        u1 = self.fc2(Gaussian_activation(self.fc1(x)))
        return u1

def cosine_activation(x): 
    return torch.cos(x)

def cosine_activation_dx(x):
    return -torch.sin(x) 

class model_cosine(nn.Module):
    """ cosine shallow neural network
    Parameters: 
    input size: input dimension
    hidden_size1 : number of hidden layers 
    num_classes: output classes 
    """
    def __init__(self, input_size, hidden_size1, num_classes,k = 1):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, num_classes,bias = False)
    def forward(self, x):
        u1 = self.fc2( cosine_activation(self.fc1(x)) )
        return u1
    

class model_tanh(nn.Module):
    """ cosine shallow neural network
    Parameters: 
    input size: input dimension
    hidden_size1 : number of hidden layers 
    num_classes: output classes 
    """
    def __init__(self, input_size, hidden_size1, num_classes,k = 1):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, num_classes,bias = False)
    def forward(self, x):
        u1 = self.fc2( F.tanh(self.fc1(x)) )
        return u1
    
def tanh_activation_dx(x): 
    return 1/torch.cosh(x)**2    


## quadrature 

In [6]:

def PiecewiseGQ2D_weights_points(Nx, order): 
    """ A slight modification of PiecewiseGQ2D function that only needs the weights and integration points.
    Parameters
    ----------

    Nx: int 
        number of intervals along the dimension. No Ny, assume Nx = Ny
    order: int 
        order of the Gauss Quadrature

    Returns
    -------
    long_weights: torch.tensor
    integration_points: torch.tensor
    """
    x, w = np.polynomial.legendre.leggauss(order)
    gauss_pts = np.array(np.meshgrid(x,x,indexing='ij')).reshape(2,-1).T
    weights =  (w*w[:,None]).ravel()

    gauss_pts =torch.tensor(gauss_pts)
    weights = torch.tensor(weights)

    h = 1/Nx # 100 intervals 
    long_weights =  torch.tile(weights,(Nx**2,1))
    long_weights = long_weights.reshape(-1,1)
    long_weights = long_weights * h**2 /4 

    integration_points = torch.tile(gauss_pts,(Nx**2,1))
    scale_factor = h/2 
    integration_points = scale_factor * integration_points

    index = np.arange(1,Nx+1)-0.5
    ordered_pairs = np.array(np.meshgrid(index,index,indexing='ij'))
    ordered_pairs = ordered_pairs.reshape(2,-1).T

    # print(ordered_pairs)
    # print()
    ordered_pairs = torch.tensor(ordered_pairs)
    # print(ordered_pairs.size())
    ordered_pairs = torch.tile(ordered_pairs, (1,order**2)) # number of GQ points

    ordered_pairs =  ordered_pairs.reshape(-1,2)
    translation = ordered_pairs*h 

    integration_points = integration_points + translation 
    return long_weights, integration_points

def MontoCarloHalton_dDim_weights_points(N = 10000, d = 2):
    Halton_gen = qmc.Halton(d, scramble=False)
    integration_points  = torch.tensor(Halton_gen.random(N))
    weights = torch.ones(N,1)/N 
    return weights, integration_points   

## Initialization
def adjust_neuron_position(my_model,target=None):
    counter = 0 
    positions = torch.tensor([[0.,0.],[0.,1.],[1.,1.],[1.,0.]])
    neuron_num = my_model.fc1.bias.size(0)
    for i in range(neuron_num): 
        w = my_model.fc1.weight.data[i:i+1,:]
        b = my_model.fc1.bias.data[i]
        values = torch.matmul(positions,w.T) # + b
        left_end = - torch.max(values)
        right_end = - torch.min(values) 
        off_set = (right_end - left_end)/1000 
        if b <= left_end + off_set: # nearly vanishing
            b = torch.rand(1)*(right_end - left_end - off_set*2) + left_end + off_set 
            my_model.fc1.bias.data[i] = b 
        if b >= right_end - off_set: # nearly nonvanishing everywhere
            if counter < 3:
                counter += 1
            else: # 3 or more 
                b = torch.rand(1)*(right_end - left_end - off_set*2) + left_end + off_set
                my_model.fc1.bias.data[i] = b 
    return my_model


def plot_l2_error_history(alg_name,hidden_size1,err,err_name = 'L2 error' ): 
    plt.figure(dpi = 100)
    plt.title(alg_name + ': neuron number '+str(hidden_size1))
    plt.plot(err, label = err_name, linewidth=1)
    plt.legend()
    plt.xlabel('epoch')
    plt.yscale('log')
    plt.ylabel('error')
    plt.show()

def plot_2D(f): 
    
    Nx = 200
    Ny = 200 
    xs = np.linspace(0, 1, Nx)
    ys = np.linspace(0, 1, Ny)
    x, y = np.meshgrid(xs, ys, indexing='xy')
    xy_comb = np.stack((x.flatten(),y.flatten())).T
    xy_comb = torch.tensor(xy_comb)
    z = f(xy_comb).reshape(Nx,Ny)
    z = z.detach().numpy()
    plt.figure(dpi=200)
    ax = plt.axes(projection='3d')
    ax.plot_surface(x , y , z )

    plt.show()


def minimize_linear_layer_H1(model,target, solver="direct",Nx = 50, order =3):
    """
    calls the following functions (dependency): 
    1. GQ_piecewise_2D
    input: the nn model containing parameter 
    1. define the loss function  
    2. take derivative to extract the linear system A
    3. call the cg solver in scipy to solve the linear system 
    output: sol. solution of Ax = b
    """

    gw_expand, integration_points = PiecewiseGQ2D_weights_points(Nx, order) 

    def loss_function_inside(x):
        model_values = model(x)
        grad_out = torch.autograd.grad(outputs =model_values, inputs = x, grad_outputs=torch.ones_like(model_values), retain_graph=True, create_graph=True)[0]         
        model_x = grad_out[:,0:1]
        model_y = grad_out[:,1:2] 
        coef = coef_torch(x)     
        return 0.5*torch.pow( model_values - target(x),2).to(device)  + 0.5* coef* torch.pow(model_x,2).to(device) + 0.5* coef * torch.pow(model_y,2).to(device) 

    def rhs_loss_inside(x): 
        # Helper function: differentiate this wrt linear layer parameters give the rhs
        return model(x)*target(x)

    integration_points.requires_grad_(True)

    loss =  gw_expand.t()@ loss_function_inside(integration_points) 
    # integration_points.requires_grad_(False)
    
    # Extract the linear system A :  
    du1 = torch.autograd.grad(outputs=loss, inputs=model.fc2.weight, retain_graph=True,create_graph = True)[0]
    neuron_number = model.fc1.bias.size(0)
    jac = torch.cat([torch.autograd.grad(outputs=du1[0,i], inputs=model.fc2.weight, retain_graph=True)[0] for i in range(neuron_number)], dim = 0)  
    

    loss_helper = gw_expand.t()@rhs_loss_inside(integration_points) 
    rhs = torch.autograd.grad(outputs=loss_helper, inputs=model.fc2.weight, retain_graph=True,create_graph = True)[0]
    rhs = rhs.view(-1,1)
    jac = jac.detach()
    rhs = rhs.detach() 
    
    # Solve the linear system using CG
    if solver == "cg": 
        sol, exit_code = linalg.cg(np.array(jac.detach().cpu()),np.array(rhs.detach().cpu()),tol=1e-12)
        sol = torch.tensor(sol).view(1,-1)
    elif solver == "direct": 
#         sol = np.linalg.inv( np.array(jac.detach().cpu()) )@np.array(rhs.detach().cpu())
        sol = (torch.linalg.solve( jac.detach(), rhs.detach())).view(1,-1)
    elif solver == "ls":
        sol = (torch.linalg.lstsq(jac.detach().cpu(),rhs.detach().cpu(),driver='gelsd').solution).view(1,-1)
        # sol = (torch.linalg.lstsq(jac.detach(),rhs.detach()).solution).view(1,-1) # gpu/cpu, driver = 'gels', cannot solve singular
    return sol 

def minimize_linear_layer_H1_explicit_assemble(model,target,solver="direct",Nx = 50, order =3,activation = 'relu' ):
    """
    calls the following functions (dependency): 
    1. GQ_piecewise_2D
    input: the nn model containing parameter 
    1. define the loss function  
    2. take derivative to extract the linear system A
    3. call the cg solver in scipy to solve the linear system 
    output: sol. solution of Ax = b
    """

    weights, integration_points = PiecewiseGQ2D_weights_points(Nx, order) 
    integration_points.requires_grad_(True) 
    start_time = time.time() 
    w = model.fc1.weight.data 
    b = model.fc1.bias.data 
    neuron_num = b.size(0) 

    if activation == 'relu':
        basis_value_col = F.relu(integration_points @ w.t()+ b)**(model.k) 
    elif activation == 'tanh': 
        basis_value_col = torch.tanh(integration_points @ w.t()+ b) 
    elif activation == 'gaussian':
        basis_value_col = Gaussian_activation(integration_points @ w.t()+ b)
    elif activation == 'cosine':
        basis_value_col = cosine_activation(integration_points @ w.t()+ b) 

    # an slow way of using auto differentitation
    basis_value_dx_col  = [] 
    basis_value_dy_col = [] 
    for i in range(neuron_num): 
        basis_value_col_i = basis_value_col[:,i] 
        grad_basis_i_value_col = torch.autograd.grad(outputs=basis_value_col_i, inputs=integration_points, grad_outputs=torch.ones_like(basis_value_col_i),retain_graph=True)[0]
        basis_value_dx_col.append(grad_basis_i_value_col [:,0:1]) 
        basis_value_dy_col.append(grad_basis_i_value_col [:,1:2]) 
    basis_value_dx_col = torch.cat(basis_value_dx_col,dim = 1) 
    basis_value_dy_col = torch.cat(basis_value_dy_col,dim = 1)  
    integration_points.requires_grad_(False) 

    weighted_basis_value_col = basis_value_col * weights 
    jac1 = weighted_basis_value_col.t() @ basis_value_col  # mass matrix 
    rhs = weighted_basis_value_col.t() @ (target(integration_points)) 
    print("assembling the mass matrix time taken: ", time.time()-start_time) 

    start_time = time.time() 
    weighted_basis_value_dx_col = basis_value_dx_col * weights
    weighted_basis_value_dy_col = basis_value_dy_col * weights
    jac2 = weighted_basis_value_dx_col.t() @ basis_value_dx_col + weighted_basis_value_dy_col.t() @ basis_value_dy_col 
    print("assembling the stiffness matrix time taken: ", time.time()-start_time)   
    jac = jac1 + jac2    
    
    start_time = time.time()    
    if solver == "cg": 
        sol, exit_code = linalg.cg(np.array(jac.detach().cpu()),np.array(rhs.detach().cpu()),tol=1e-12)
        sol = torch.tensor(sol).view(1,-1)
    elif solver == "direct": 
#         sol = np.linalg.inv( np.array(jac.detach().cpu()) )@np.array(rhs.detach().cpu())
        sol = (torch.linalg.solve( jac.detach(), rhs.detach())).view(1,-1)
    elif solver == "ls":
        sol = (torch.linalg.lstsq(jac.detach().cpu(),rhs.detach().cpu(),driver='gelsd').solution).view(1,-1)
        # sol = (torch.linalg.lstsq(jac.detach(),rhs.detach()).solution).view(1,-1) # gpu/cpu, driver = 'gels', cannot solve singular
    print("solving Ax = b time taken: ", time.time()-start_time)
    return sol 

def minimize_linear_layer_H1_explicit_assemble_efficient(model,target,solver="direct",Nx = 50, order =3,activation = 'relu' ):

    weights, integration_points = PiecewiseGQ2D_weights_points(Nx, order) 
    integration_points.requires_grad_(True) 
    start_time = time.time() 
    w = model.fc1.weight.data 
    b = model.fc1.bias.data 
    neuron_num = b.size(0) 

    if activation == 'relu':
        basis_value_col = F.relu(integration_points @ w.t()+ b)**(model.k) 
        if model.k == 1:  
            basis_value_dx_col = torch.heaviside(integration_points @ w.t()+ b, torch.tensor([0.])) * w.t()[0:1,:] 
            basis_value_dy_col = torch.heaviside(integration_points @ w.t()+ b,torch.tensor([0.])) * w.t()[1:2,:] 
        else: 
            basis_value_dx_col = model.k * F.relu(integration_points @ w.t()+ b)**(model.k-1) * w.t()[0:1,:]
            basis_value_dy_col = model.k * F.relu(integration_points @ w.t()+ b)**(model.k-1) * w.t()[1:2,:] 
    elif activation == 'tanh': 
        basis_value_col = torch.tanh(integration_points @ w.t()+ b) 
        basis_value_dx_col = tanh_activation_dx(integration_points @ w.t()+ b) * w.t()[0:1,:]
        basis_value_dy_col = tanh_activation_dx(integration_points @ w.t()+ b) * w.t()[1:2,:]
    elif activation == 'gaussian':
        basis_value_col = Gaussian_activation(integration_points @ w.t()+ b)
        basis_value_dx_col = Gaussian_activation_dx(integration_points @ w.t()+ b) * w.t()[0:1,:]
        basis_value_dy_col = Gaussian_activation_dx(integration_points @ w.t()+ b) * w.t()[1:2,:]
    elif activation == 'cosine':
        basis_value_col = cosine_activation(integration_points @ w.t()+ b) 
        basis_value_dx_col = cosine_activation_dx(integration_points @ w.t()+ b) * w.t()[0:1,:]
        basis_value_dy_col = cosine_activation_dx(integration_points @ w.t()+ b) * w.t()[1:2,:] 

    weighted_basis_value_col = basis_value_col * weights 
    jac1 = weighted_basis_value_col.t() @ basis_value_col  # mass matrix 
    rhs = weighted_basis_value_col.t() @ (target(integration_points)) 
    print("assembling the mass matrix time taken: ", time.time()-start_time) 

    start_time = time.time() 
    weighted_basis_value_dx_col = basis_value_dx_col * weights
    weighted_basis_value_dy_col = basis_value_dy_col * weights
    jac2 = weighted_basis_value_dx_col.t() @ basis_value_dx_col + weighted_basis_value_dy_col.t() @ basis_value_dy_col 
    print("assembling the stiffness matrix time taken: ", time.time()-start_time)   
    jac = jac1 + jac2    
    
    start_time = time.time()    
    if solver == "cg": 
        sol, exit_code = linalg.cg(np.array(jac.detach().cpu()),np.array(rhs.detach().cpu()),tol=1e-12)
        sol = torch.tensor(sol).view(1,-1)
    elif solver == "direct": 
#         sol = np.linalg.inv( np.array(jac.detach().cpu()) )@np.array(rhs.detach().cpu())
        sol = (torch.linalg.solve( jac.detach(), rhs.detach())).view(1,-1)
    elif solver == "ls":
        sol = (torch.linalg.lstsq(jac.detach().cpu(),rhs.detach().cpu(),driver='gelsd').solution).view(1,-1)
        # sol = (torch.linalg.lstsq(jac.detach(),rhs.detach()).solution).view(1,-1) # gpu/cpu, driver = 'gels', cannot solve singular
    print("solving Ax = b time taken: ", time.time()-start_time)
    return sol 




## Compute jacobian and rhs 

In [7]:

def compute_jac_rhs_H1(model,target, solver="direct",Nx = 50, order =3):
    """
    calls the following functions (dependency): 
    1. GQ_piecewise_2D
    input: the nn model containing parameter 
    1. define the loss function  
    2. take derivative to extract the linear system A
    3. call the cg solver in scipy to solve the linear system 
    output: sol. solution of Ax = b
    """

    gw_expand, integration_points = PiecewiseGQ2D_weights_points(Nx, order) 

    def loss_function_inside(x):
        model_values = model(x)
        grad_out = torch.autograd.grad(outputs =model_values, inputs = x, grad_outputs=torch.ones_like(model_values), retain_graph=True, create_graph=True)[0]         
        model_x = grad_out[:,0:1]
        model_y = grad_out[:,1:2] 
        coef = coef_torch(x)     
        return 0.5*torch.pow( model_values - target(x),2).to(device)  + 0.5* coef* torch.pow(model_x,2).to(device) + 0.5* coef * torch.pow(model_y,2).to(device) 

    def rhs_loss_inside(x): 
        # Helper function: differentiate this wrt linear layer parameters give the rhs
        return model(x)*target(x)

    integration_points.requires_grad_(True)

    loss =  gw_expand.t()@ loss_function_inside(integration_points) 
    # integration_points.requires_grad_(False)
    
    # Extract the linear system A :  
    du1 = torch.autograd.grad(outputs=loss, inputs=model.fc2.weight, retain_graph=True,create_graph = True)[0]
    neuron_number = model.fc1.bias.size(0)
    jac = torch.cat([torch.autograd.grad(outputs=du1[0,i], inputs=model.fc2.weight, retain_graph=True)[0] for i in range(neuron_number)], dim = 0)  
    

    loss_helper = gw_expand.t()@rhs_loss_inside(integration_points) 
    rhs = torch.autograd.grad(outputs=loss_helper, inputs=model.fc2.weight, retain_graph=True,create_graph = True)[0]
    rhs = rhs.view(-1,1)
    jac = jac.detach()
    rhs = rhs.detach() 
    
    return jac,rhs 

def compute_jac_rhs_H1_explicit_assemble(model,target,solver="direct",Nx = 50, order =3,activation = 'relu' ):
    """
    calls the following functions (dependency): 
    1. GQ_piecewise_2D
    input: the nn model containing parameter 
    1. define the loss function  
    2. take derivative to extract the linear system A
    3. call the cg solver in scipy to solve the linear system 
    output: sol. solution of Ax = b
    """

    weights, integration_points = PiecewiseGQ2D_weights_points(Nx, order) 
    integration_points.requires_grad_(True) 
    start_time = time.time() 
    w = model.fc1.weight.data 
    b = model.fc1.bias.data 
    neuron_num = b.size(0) 

    if activation == 'relu':
        basis_value_col = F.relu(integration_points @ w.t()+ b)**(model.k) 
    elif activation == 'tanh': 
        basis_value_col = torch.tanh(integration_points @ w.t()+ b) 
    elif activation == 'gaussian':
        basis_value_col = Gaussian_activation(integration_points @ w.t()+ b)
    elif activation == 'cosine':
        basis_value_col = cosine_activation(integration_points @ w.t()+ b) 

    # an slow way of using auto differentitation
    basis_value_dx_col  = [] 
    basis_value_dy_col = [] 
    for i in range(neuron_num): 
        basis_value_col_i = basis_value_col[:,i] 
        grad_basis_i_value_col = torch.autograd.grad(outputs=basis_value_col_i, inputs=integration_points, grad_outputs=torch.ones_like(basis_value_col_i),retain_graph=True)[0]
        basis_value_dx_col.append(grad_basis_i_value_col [:,0:1]) 
        basis_value_dy_col.append(grad_basis_i_value_col [:,1:2]) 
    basis_value_dx_col = torch.cat(basis_value_dx_col,dim = 1) 
    basis_value_dy_col = torch.cat(basis_value_dy_col,dim = 1)  
    integration_points.requires_grad_(False) 

    weighted_basis_value_col = basis_value_col * weights 
    jac1 = weighted_basis_value_col.t() @ basis_value_col  # mass matrix 
    rhs = weighted_basis_value_col.t() @ (target(integration_points)) 
    print("assembling the mass matrix time taken: ", time.time()-start_time) 

    start_time = time.time() 
    weighted_basis_value_dx_col = basis_value_dx_col * weights
    weighted_basis_value_dy_col = basis_value_dy_col * weights
    jac2 = weighted_basis_value_dx_col.t() @ basis_value_dx_col + weighted_basis_value_dy_col.t() @ basis_value_dy_col 
    print("assembling the stiffness matrix time taken: ", time.time()-start_time)   
    jac = jac1 + jac2    
    
    return jac, rhs 

def compute_jac_rhs_H1_explicit_assemble_efficient(model,target,solver="direct",Nx = 50, order =3,activation = 'relu' ):

    weights, integration_points = PiecewiseGQ2D_weights_points(Nx, order) 
    integration_points.requires_grad_(True) 
    start_time = time.time() 
    w = model.fc1.weight.data 
    b = model.fc1.bias.data 
    neuron_num = b.size(0) 

    if activation == 'relu':
        basis_value_col = F.relu(integration_points @ w.t()+ b)**(model.k) 
        if model.k == 1:  
            basis_value_dx_col = torch.heaviside(integration_points @ w.t()+ b, torch.tensor([0.])) * w.t()[0:1,:] 
            basis_value_dy_col = torch.heaviside(integration_points @ w.t()+ b,torch.tensor([0.])) * w.t()[1:2,:] 
        else: 
            basis_value_dx_col = model.k * F.relu(integration_points @ w.t()+ b)**(model.k-1) * w.t()[0:1,:]
            basis_value_dy_col = model.k * F.relu(integration_points @ w.t()+ b)**(model.k-1) * w.t()[1:2,:] 
    elif activation == 'tanh': 
        basis_value_col = torch.tanh(integration_points @ w.t()+ b) 
        basis_value_dx_col = tanh_activation_dx(integration_points @ w.t()+ b) * w.t()[0:1,:]
        basis_value_dy_col = tanh_activation_dx(integration_points @ w.t()+ b) * w.t()[1:2,:]
    elif activation == 'gaussian':
        basis_value_col = Gaussian_activation(integration_points @ w.t()+ b)
        basis_value_dx_col = Gaussian_activation_dx(integration_points @ w.t()+ b) * w.t()[0:1,:]
        basis_value_dy_col = Gaussian_activation_dx(integration_points @ w.t()+ b) * w.t()[1:2,:]
    elif activation == 'cosine':
        basis_value_col = cosine_activation(integration_points @ w.t()+ b) 
        basis_value_dx_col = cosine_activation_dx(integration_points @ w.t()+ b) * w.t()[0:1,:]
        basis_value_dy_col = cosine_activation_dx(integration_points @ w.t()+ b) * w.t()[1:2,:] 

    weighted_basis_value_col = basis_value_col * weights 
    jac1 = weighted_basis_value_col.t() @ basis_value_col  # mass matrix 
    rhs = weighted_basis_value_col.t() @ (target(integration_points)) 
    print("assembling the mass matrix time taken: ", time.time()-start_time) 

    start_time = time.time() 
    weighted_basis_value_dx_col = basis_value_dx_col * weights
    weighted_basis_value_dy_col = basis_value_dy_col * weights
    jac2 = weighted_basis_value_dx_col.t() @ basis_value_dx_col + weighted_basis_value_dy_col.t() @ basis_value_dy_col 
    print("assembling the stiffness matrix time taken: ", time.time()-start_time)   
    jac = jac1 + jac2    
    
    return jac, rhs 





### verify the three functions give the same jacobian and rhs

In [13]:

def target(x):
    return torch.cos(pi * x[:,0:1]) * torch.cos(pi * x[:,1:2])
    # return torch.cos(pi * x[:,0:1])  + torch.cos(pi * x[:,1:2])
    
def rhs(x):
    return (1 + 2 * pi**2) * torch.cos(pi * x[:,0:1]) * torch.cos(pi * x[:,1:2]) 

# my_model = model(2, 100, 1, k = 3)
my_model = model_gaussian(2, 80, 1) 
my_model = adjust_neuron_position(my_model)
# jac1, rhs1_vec = compute_jac_rhs_H1(my_model,rhs,solver="direct",Nx = 50, order =3 )
# jac2, rhs2_vec = compute_jac_rhs_H1_explicit_assemble(my_model,rhs,solver="direct",Nx = 50, order =3,activation = 'gaussian' )
jac3, rhs3_vec = compute_jac_rhs_H1_explicit_assemble_efficient(my_model,rhs,solver="direct",Nx = 50, order =3,activation = 'gaussian' )

# print(torch.allclose(jac1,jac2))
# print(torch.allclose(jac1,jac3))
# print(torch.allclose(rhs1_vec,rhs2_vec))
# print(torch.allclose(rhs1_vec,rhs3_vec)) 



assembling the mass matrix time taken:  0.3317549228668213
assembling the stiffness matrix time taken:  0.007397890090942383
assembling the mass matrix time taken:  0.018674850463867188
assembling the stiffness matrix time taken:  0.004733085632324219
True
True
True
True


In [26]:
# my_model = model(2, 200, 1, k = 1)

my_model = model_gaussian(2, 100, 1) 
my_model = adjust_neuron_position(my_model)
my_model = weights_init_uniform_non_relu(my_model,1)  
# jac1, rhs1_vec = compute_jac_rhs_H1(my_model,rhs,solver="direct",Nx = 50, order =3 )
# jac2, rhs2_vec = compute_jac_rhs_H1_explicit_assemble(my_model,rhs,solver="direct",Nx = 50, order =3,activation = 'gaussian' )
jac3, rhs3_vec = compute_jac_rhs_H1_explicit_assemble_efficient(my_model,rhs,solver="direct",Nx = 50, order =3,activation = 'gaussian' )

print(torch.linalg.eigvals(jac3))
min_val = torch.real(torch.linalg.eigvals(jac3)[-1])
max_val = torch.real(torch.linalg.eigvals(jac3)[0])
print("cond", max_val/min_val)


assembling the mass matrix time taken:  0.11319112777709961
assembling the stiffness matrix time taken:  0.03450202941894531
tensor([ 5.1093e+01+0.0000e+00j,  1.2298e+01+0.0000e+00j,
         1.3684e+01+0.0000e+00j,  1.7443e+00+0.0000e+00j,
         1.2430e+00+0.0000e+00j,  5.2464e-01+0.0000e+00j,
         1.5921e-01+0.0000e+00j,  1.2083e-01+0.0000e+00j,
         2.5427e-02+0.0000e+00j,  1.6291e-02+0.0000e+00j,
         1.0712e-02+0.0000e+00j,  9.4273e-03+0.0000e+00j,
         2.5722e-03+0.0000e+00j,  5.8613e-04+0.0000e+00j,
         4.8832e-04+0.0000e+00j,  2.3494e-04+0.0000e+00j,
         1.8246e-04+0.0000e+00j,  7.2296e-05+0.0000e+00j,
         4.2663e-05+0.0000e+00j,  1.0142e-05+0.0000e+00j,
         6.8842e-06+0.0000e+00j,  4.0519e-06+0.0000e+00j,
         2.5227e-06+0.0000e+00j,  1.7545e-06+0.0000e+00j,
         7.0718e-07+0.0000e+00j,  5.9763e-07+0.0000e+00j,
         2.3678e-07+0.0000e+00j,  1.5869e-07+0.0000e+00j,
         5.1240e-08+0.0000e+00j,  4.3406e-08+0.0000e+00j,
     

## RFM algorithm 

In [9]:


def RFM_H1_train(my_model,target,rhs, gw_expand, integration_points, initialize_,plot = True): 

    def l2_loss_function_inside(x):
        return 0.5*torch.pow(my_model(x)-target(x),2).to(device)

    def h10_loss_function_inside(x):
        target_values = target(x) 
        target_grad = torch.autograd.grad(outputs=target_values, inputs=x, grad_outputs= torch.ones_like(target_values),retain_graph=True, create_graph=True)[0] 
        target_x = target_grad[:,0:1]
        target_y = target_grad[:,1:2] 

        model_values = my_model(x)
        # model_grad = torch.autograd.grad(outputs=model_values, inputs=x, grad_outputs=torch.ones_like(model_values),retain_graph=True,create_graph = True)[0]
        model_grad = torch.autograd.grad(outputs=model_values, inputs=x, grad_outputs=torch.ones_like(model_values))[0] # no need to create or retain_graph, save memory 
        model_x = model_grad[:,0:1]
        model_y = model_grad[:,1:2] 
        return (model_x - target_x)**2 +  (model_y - target_y)**2
    
    start = time.time()
    hidden_size1 = my_model.fc1.bias.size(0)
    print("current hidden layer size: ",hidden_size1)
        
    # Data to be recorded
    # Parameter initialization
    if initialize_ != None: 
        initialize_(my_model,target)
    with torch.no_grad():
        half_l2_err_sqrd = gw_expand.t()@l2_loss_function_inside(integration_points)
    l2_err = (half_l2_err_sqrd*2)**0.5
    integration_points.requires_grad_(True) 
    # energy_arr[0] = (gw_expand.t()@total_loss_function_inside(integration_points)).detach() 
    # exact_energy = (gw_expand.t()@exact_energy_function_inside(integration_points)).detach()
    integration_points.requires_grad_(False) 

    # print("original error: {} \t energy {}".format(l2_err) )
    
    # my_model = adjust_neuron_position(my_model) 
    # a-minimization 
    for param in my_model.fc2.parameters():
        param.requires_grad_(True) 
    for param in my_model.fc1.parameters():
        param.requires_grad_(False) 
    sol = minimize_linear_layer_H1(my_model,rhs,'direct')
    my_model.fc2.weight.data[0,:] = sol[:]
    if plot: 
        plot_2D(my_model.cpu())

    with torch.no_grad():
        half_l2_err_sqrd = gw_expand.t()@l2_loss_function_inside(integration_points) 
        l2_err = ((half_l2_err_sqrd * 2 )**0.5).detach() 
    integration_points.requires_grad_(True) 
    h10_err = (gw_expand.t() @ h10_loss_function_inside(integration_points))**0.5 
    integration_points.requires_grad_(False) 
    # Plot 1. L^2 error history
    if plot: 
        plot_2D(my_model.cpu()) 
    end = time.time()
    print(str(end - start)+" s")
    
    return l2_err, h10_err, my_model

def coef_torch(x): 
    return 1.0  

def target(x):
    return torch.cos(pi * x[:,0:1]) * torch.cos(pi * x[:,1:2])
    # return torch.cos(pi * x[:,0:1])  + torch.cos(pi * x[:,1:2])
    
def rhs(x):
    return (1 + 2 * pi**2) * torch.cos(pi * x[:,0:1]) * torch.cos(pi * x[:,1:2]) 

# gw_expand, integration_points = MontoCarloHalton_dDim_weights_points(10000, 2)
gw_expand, integration_points = PiecewiseGQ2D_weights_points(50, 3) 




In [8]:

def target(x):
    return torch.cos(pi * x[:,0:1]) * torch.cos(pi * x[:,1:2])
    # return torch.cos(pi * x[:,0:1])  + torch.cos(pi * x[:,1:2])
    
def rhs(x):
    return (1 + 2 * pi**2) * torch.cos(pi * x[:,0:1]) * torch.cos(pi * x[:,1:2]) 

# my_model = model(2, 200, 1, k = 3)
my_model = model_tanh(2, 200, 1) 
my_model = adjust_neuron_position(my_model)
sol = minimize_linear_layer_H1_explicit_assemble_efficient(my_model,rhs,solver="direct",Nx = 50, order =3,activation = 'tanh' )
# sol = minimize_linear_layer_H1(my_model,rhs,solver="direct",Nx = 50, order =3) 
my_model.fc2.weight.data[0,:] = sol[:]

weights, integration_points = PiecewiseGQ2D_weights_points(50, 3) 
l2_err = ( weights.t() @ (my_model(integration_points) - target(integration_points))**2 )**0.5 

print(l2_err)   


assembling the mass matrix time taken:  0.08005094528198242
assembling the stiffness matrix time taken:  0.03822493553161621
solving Ax = b time taken:  0.0017919540405273438
tensor([[2.1185e-05]], grad_fn=<PowBackward0>)


In [9]:
if __name__ == "__main__":

    gw_expand, integration_points = PiecewiseGQ2D_weights_points(50, 3)  
    gw_expand.requires_grad_(True)
    integration_points.requires_grad_(True) 
    initialize_ = None # adjust_neuron_position 
    plot = False 
    save = False 

    num_trials  = 1  
    for exponent in [2,3,4,5,6,7,8]: # 5,6,7,8  
        print("neuron number: ", 2**exponent)
        for trial in range(num_trials): 
            my_model = model_tanh(2,2**exponent,1)
            # my_model = model(2,2**exponent,3)
            my_model = weights_init_uniform_non_relu(my_model,2) 
            l2_err, h10_err, my_model = RFM_H1_train(my_model, \
                            target,rhs,gw_expand, integration_points, initialize_,plot = plot)
            print("\t trial {}: l2_err {}, h10_err {}".format(trial, l2_err ,h10_err))
            if save:
                torch.save(my_model.state_dict(),  "LSGD-H1-"+str(exponent)+"-"+str(trial)+".pt")
                torch.save(l2_err,  "LSGD-H1-"+str(exponent)+"-"+str(trial)+"-err.pt")
                torch.save(h10_err, "LSGD-H1-"+str(exponent)+"-"+str(trial)+"-energy_arr.pt")
        print()     

neuron number:  4
current hidden layer size:  4
0.11189579963684082 s
	 trial 0: l2_err tensor([[0.4467]]), h10_err tensor([[1.8694]], grad_fn=<PowBackward0>)

neuron number:  8
current hidden layer size:  8
0.05100202560424805 s
	 trial 0: l2_err tensor([[0.1493]]), h10_err tensor([[1.0373]], grad_fn=<PowBackward0>)

neuron number:  16
current hidden layer size:  16
0.13719415664672852 s
	 trial 0: l2_err tensor([[0.0244]]), h10_err tensor([[0.2872]], grad_fn=<PowBackward0>)

neuron number:  32
current hidden layer size:  32
0.2582111358642578 s
	 trial 0: l2_err tensor([[0.0007]]), h10_err tensor([[0.0145]], grad_fn=<PowBackward0>)

neuron number:  64
current hidden layer size:  64
0.5629370212554932 s
	 trial 0: l2_err tensor([[1.4140e-05]]), h10_err tensor([[0.0004]], grad_fn=<PowBackward0>)

neuron number:  128
current hidden layer size:  128
1.5400900840759277 s
	 trial 0: l2_err tensor([[1.6482e-06]]), h10_err tensor([[5.7303e-05]], grad_fn=<PowBackward0>)

neuron number:  256
c